# ATLAS Open Event Generation: phenomenology with Rivet

ATLAS has released a large amount of [event generator output](https://opendata.atlas.cern/docs/data/for_research/evgen_data)
as open data, in HepMC format. This notebook shows how to take one of those samples and analyse it with
[Rivet](https://rivet.hepforge.org/), the standard toolkit for confronting generator-level events with
physics measurements, which is a common starting point for phenomenology studies.

Here's what we'll do:

1. Find a sample and read its metadata with [atlasopenmagic](https://opendata.atlas.cern/docs/atlasopenmagic).
2. Get one HepMC event file to work with.
3. Run a standard Rivet analysis (`MC_ZINC`, inclusive Z / Drell-Yan) over the events.
4. Look at the results, both as a `rivet-mkhtml` gallery and inline in the notebook.
5. Write and run our own small Rivet analysis.
6. Save the histograms for later use.

We'll use dataset **601189** (a POWHEG+Pythia8 Drell-Yan $Z\to ee$ sample at $\sqrt{s}=13.6$ TeV) as our
running example, but every step is written so you can swap in any other dataset.

> **What you need:** this notebook assumes it is running somewhere with **CVMFS** mounted, so that Rivet,
> YODA and HepMC, along with the `atlasopenmagic` package, are available from the software stack. CERN's
> SWAN service is one way to get such an environment (see [What is SWAN?](https://swan.docs.cern.ch/intro/what_is/)),
> and any machine with CVMFS will do.

## Getting set up

We just need Rivet, YODA and HepMC on the path. The CVMFS software stack provides them (on SWAN, pick a
recent LCG release when you start your session). Let's check they are there:

In [ ]:
# --- Environment sanity check (run in a SWAN terminal or here with the ! shell escape) ---
!echo "Rivet:"; rivet --version 2>/dev/null || echo "  rivet not found - load an LCG stack that provides it"
!echo "YODA:";  python3 -c "import yoda; print(' ', yoda.version())" 2>/dev/null || yodals --version 2>/dev/null || echo "  yoda not found"
!echo "HepMC:"; HepMC3-config --version 2>/dev/null || echo "  HepMC not found - load an LCG stack that provides it"

## 1. Finding a sample with `atlasopenmagic`

[`atlasopenmagic`](https://opendata.atlas.cern/docs/atlasopenmagic) is the helper that knows about all the
ATLAS Open Data samples: their metadata (cross-section, generator, keywords, ...) and where the files
live. It comes from CVMFS along with everything else, so there is nothing to install. The cell below just
makes it importable (loading it from the CVMFS distribution, and only falling back to `pip` if CVMFS
isn't mounted). It needs a Python ≥ 3.10 kernel, which any recent LCG stack provides.

In [ ]:
# Preferred on CERN SWAN / ESCAPE VRE: load atlasopenmagic from the CVMFS distribution (no pip needed).
# Requires a Python >= 3.10 kernel (pick a recent LCG / dev stack in the SWAN configuration form).
AOM_CVMFS = "/cvmfs/sw.escape.eu/atlasopenmagic/latest"

import os, sys
try:
    import atlasopenmagic  # already importable? (e.g. provided by the LCG view)
except ImportError:
    if os.path.isdir(AOM_CVMFS):
        sys.path.insert(0, AOM_CVMFS)                 # so we can import the setup helper
        from setup_notebook import setup_atlasopenmagic
        setup_atlasopenmagic(AOM_CVMFS)               # adds <home>/lib/pythonX.Y/site-packages and <home>/bin
    else:
        !{sys.executable} -m pip install --user atlasopenmagic   # fallback if CVMFS is not available

import atlasopenmagic as atom
print("Using atlasopenmagic from:", atom.__file__)

In [ ]:
# See what open-data releases exist
atom.available_releases()

We'll use the event-generation release at 13.6 TeV (there's a 13 TeV one too, `2025r-evgen-13tev`).
Unlike the reconstructed-data releases, these hold generator-level **HepMC** events, which is what Rivet
reads.

In [ ]:
RELEASE = "2025r-evgen-13p6tev"   # or "2025r-evgen-13tev"
atom.set_release(RELEASE)
print("Active release:", atom.get_current_release())

### Browsing by physics

Every sample is tagged with keywords (`z`, `top`, `higgs`, `diboson`, ...). Let's list them, then pick out
the Drell-Yan $Z\to ee$ samples.

In [ ]:
# All keywords available in this release
print(atom.available_keywords())

# Find Z->ee samples by their short physics name
atom.match_metadata("physics_short", "Zee")

### A look at our sample

We'll go with **601189** (`PhPy8EG_AZNLO_Zee`). Its metadata has everything we need to normalise the
sample later: the cross-section, filter efficiency, k-factor and number of generated events.

In [ ]:
DSID = "601189"   # PhPy8EG_AZNLO_Zee  (POWHEG+Pythia8 Drell-Yan Z->ee, sqrt(s)=13.6 TeV)

meta = atom.get_metadata(DSID)
for k in ["physics_short", "description", "generator", "GenTune",
          "CoMEnergy", "cross_section_pb", "kFactor", "genFiltEff",
          "nEvents", "hepmc_version", "keywords"]:
    print(f"{k:20s}: {meta.get(k)}")

# Effective cross-section for normalisation (pb):  sigma * k-factor * filter-eff
xsec_pb = float(meta["cross_section_pb"]) * float(meta["kFactor"]) * float(meta["genFiltEff"])
print("\nEffective cross-section [pb]:", xsec_pb)

## 2. Locating the files

`get_urls` hands back the list of HepMC files for a dataset. You can ask for whichever access protocol
suits you:

- `https`: a plain web download, works anywhere;
- `root`: XRootD streaming;
- `eos`: a POSIX `/eos/...` path you can read directly when the files are mounted (as they are on SWAN).

We'll take the first file from the list.

In [ ]:
urls_https = atom.get_urls(DSID, protocol="https")
urls_eos   = atom.get_urls(DSID, protocol="eos")

print(f"{len(urls_https)} files in dataset {DSID}\n")
print("First HTTPS URL:\n ", urls_https[0])
print("\nFirst EOS path (use this directly on SWAN):\n ", urls_eos[0])

## 3. Getting one event file

Rivet reads gzip-compressed HepMC directly, so we prefer not to leave a multi-GB plain-text file on disk.
The one catch is that the open-data files are gzipped **tar archives** (`HEPMC.*.tar.gz`) that each wrap a
*single* HepMC record, and Rivet cannot read a tar archive. So the only processing we do here is to pull
that record out of the tarball and keep it gzipped as `*.hepmc.gz`, ready for Rivet.

If the files are already mounted (e.g. on EOS via `protocol="eos"`), you can point straight at them and
skip the download; otherwise we fetch one over HTTPS. One file is enough to get started; add more for
higher statistics.

In [ ]:
import os, tarfile, gzip, shutil, urllib.request

workdir = os.path.abspath("evgen_work")
os.makedirs(workdir, exist_ok=True)

def plain_url(u):
    # atlasopenmagic returns fsspec chained URLs, e.g. "simplecache::https://...".
    # Keep only the real transport URL (the part after the last "::").
    return u.split("::")[-1]

def tar_to_hepmc_gz(tar_path, dest_dir):
    # Some datasets directly provide gzip files without tarring
    if ".tar" not in tar_path and ".tgz" not in tar_path:
        # In that case give back the file path directly
        return tar_path

    # Pull the single HepMC record out of a .tar.gz archive and store it as a gzipped
    # HepMC file (Rivet reads .gz directly). Streamed in one pass -> no large temp file.
    with tarfile.open(tar_path, "r:gz") as tf:
        member = next(m for m in tf.getmembers() if m.isfile())
        out = os.path.join(dest_dir, os.path.basename(member.name))
        if not out.endswith(".gz"):
            out += ".gz"
        with tf.extractfile(member) as src, gzip.open(out, "wb") as dst:
            shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    return out

# ---- Get the tarball --------------------------------------------------------
# On SWAN you can read straight from EOS instead of downloading:
#     tar_path = plain_url(urls_eos[0])          # /eos/opendata/atlas/rucio/.../HEPMC.*.tar.gz.1
# Off SWAN, download one file over HTTPS:
src = plain_url(urls_https[0])
tar_path = os.path.join(workdir, os.path.basename(src))
if not os.path.exists(tar_path):
    print("Downloading", src, "...")
    urllib.request.urlretrieve(src, tar_path)

HEPMC_FILE = tar_to_hepmc_gz(tar_path, workdir)
print("HepMC ready (gzipped):", HEPMC_FILE,
      f"({os.path.getsize(HEPMC_FILE)/1e6:.1f} MB compressed)")

## 4. Running a standard Rivet analysis (`MC_ZINC`)

`MC_ZINC` is Rivet's built-in inclusive-Z (Drell-Yan) analysis, a good match for our $Z\to ee$ sample.
It fills distributions like the Z transverse momentum, rapidity and mass. Rivet picks up the cross-section
from the HepMC record, and we pass it explicitly as well. Our input is the gzipped `*.hepmc.gz` from the
previous step, which Rivet decompresses on the fly.

> **Tip (skip the temp file entirely):** on a recent Rivet you can pipe the events straight in and avoid
> writing anything to disk: `!tar -xzOf "{tar_path}" | rivet /dev/stdin -a MC_ZINC --cross-section={xsec_pb} -o out.yoda.gz`

In [ ]:
YODA_OUT = os.path.join(workdir, "Zee_MC_ZINC.yoda.gz")

# --cross-section is in pb; -a selects the analysis; -o is the output histogram file
!rivet "{HEPMC_FILE}" -a MC_ZINC --cross-section={xsec_pb} -o "{YODA_OUT}"
print("\nWrote:", YODA_OUT)

## 5. Turning the histograms into plots

`rivet-mkhtml` turns one (or several) YODA files into a small browsable web page of plots: a
`rivet-plots/` folder with an `index.html` inside.

In [ ]:
PLOTDIR = os.path.join(workdir, "rivet-plots")

# rivet-mkhtml renders labels with LaTeX. Some SWAN TeX Live setups choke on it
# (e.g. "latex was not able to process ... 'lp'"). Best-effort: tell matplotlib not
# to use LaTeX for the subprocess plot scripts. If it still fails, use the inline
# plot in the next cell (identical physics, no LaTeX) - the .yoda.gz is already written.
import os
_rc = os.path.join(workdir, "matplotlibrc")
open(_rc, "w").write("text.usetex: False\n")
os.environ["MATPLOTLIBRC"] = _rc

!rivet-mkhtml "{YODA_OUT}" -o "{PLOTDIR}" || echo "rivet-mkhtml hit an error (likely LaTeX) - see the inline plot below."
print("If it succeeded, open in the SWAN file browser:", os.path.join(PLOTDIR, "index.html"))

You can open `rivet-plots/index.html` from the Jupyter file browser. The label rendering in `rivet-mkhtml`
leans on LaTeX, which isn't always fully set up, so the cell below is a dependable way to see the results
inline instead: it reads the YODA output with the `yoda` Python bindings and draws it with `matplotlib`.
It works with both YODA 2.x and older YODA 1.x.

In [ ]:
import yoda
import numpy as np
import matplotlib.pyplot as plt   # SWAN shows figures inline by default

aos = yoda.read(YODA_OUT)

def pick(suffix):
    # Return the (nominal) Histo1D whose path ends with `suffix`.
    for k, v in aos.items():
        if k.endswith(suffix) and isinstance(v, yoda.Histo1D):
            return v
    return None

def hist_xy(h):
    # Bin centres, heights (dsigma/dx) and errors - robust across YODA versions.
    # YODA 2.x: vectorised accessors on the histogram itself
    try:
        y = np.asarray(h.heights(), dtype=float)
        x = np.asarray(h.xMids() if hasattr(h, "xMids")
                       else [b.xMid() for b in h.bins()], dtype=float)
        e = np.asarray(h.heightErrs(), dtype=float) if hasattr(h, "heightErrs") \
            else np.zeros_like(y)
        return x, y, e
    except Exception:
        pass
    # YODA 1.x: per-bin accessors
    try:
        x = np.array([b.xMid()      for b in h.bins()], dtype=float)
        y = np.array([b.height()    for b in h.bins()], dtype=float)
        e = np.array([b.heightErr() for b in h.bins()], dtype=float)
        return x, y, e
    except Exception:
        pass
    # Last resort: convert to a Scatter and read the points
    s = h.mkScatter()
    pts = s.points()
    x = np.array([p.x() for p in pts]); y = np.array([p.y() for p in pts])
    try:
        e = np.array([max(abs(v) for v in p.yErrs()) for p in pts])
    except Exception:
        e = np.zeros_like(y)
    return x, y, e

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (suffix, xlabel, logy) in zip(
        axes, [("/Z_mass", r"$m_{ee}$ [GeV]", False),
               ("/Z_pT",   r"$p_{T}^{Z}$ [GeV]", True)]):
    h = pick(suffix)
    if h is None:
        ax.set_title(f"(no {suffix} in output)"); continue
    x, y, ye = hist_xy(h)
    ax.errorbar(x, y, yerr=ye, fmt="o", ms=3, lw=1)
    if logy: ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"d$\sigma$/d$x$ [pb/GeV]")
    ax.set_title(f"MC_ZINC {suffix.strip('/')}  (DSID {DSID})")
fig.tight_layout()
plt.show()

## 6. Writing your own Rivet analysis

When the built-in analyses aren't quite what you want, you can write your own. Everything happens right
here in your session, with no copying of files. Any of these works:

- **Scaffold a template:** `rivet-mkanalysis My_Analysis` creates `My_Analysis.{cc,info,plot}`; edit
  `My_Analysis.cc` in the Jupyter editor (the next cell does this).
- **Write it from the notebook:** put your C++ in a cell that starts with `%%writefile My_Analysis.cc`.
- **Bring your own:** upload an existing `My_Analysis.cc` with the Jupyter file-browser **Upload** button.

Then we build it into a plugin and point Rivet at it:

In [ ]:
# Scaffold a template routine (creates My_Analysis.{cc,info,plot}) - skip if you already uploaded yours
ANADIR = os.path.join(workdir, "my_analysis")
os.makedirs(ANADIR, exist_ok=True)
os.chdir(ANADIR)
if not os.path.exists("My_Analysis.cc"):
    !rivet-mkanalysis My_Analysis

# Build the plugin -> produces RivetMy_Analysis.so
!rivet-buildplugin My_Analysis.cc

# Run it: RIVET_ANALYSIS_PATH must include the dir with the built .so
env = f'RIVET_ANALYSIS_PATH="{ANADIR}:$RIVET_ANALYSIS_PATH"'
!{env} rivet "{HEPMC_FILE}" -a My_Analysis --cross-section={xsec_pb} -o "{os.path.join(workdir,'My_Analysis.yoda.gz')}"
os.chdir(workdir)

## 7. Taking your results with you

The histograms live in the `*.yoda.gz` files. To use them elsewhere you can:

- download the `.yoda.gz` (or the whole `rivet-plots/` folder) from the Jupyter file browser;
- convert to ROOT with `yoda2root`, for a ROOT-based analysis;
- read them back in Python (as we did above) and export to `numpy`/CSV/plots.

From here they're ready to serve as a reference or background template in your own phenomenology study.

In [ ]:
# List the artefacts produced, and optionally convert the main result to a ROOT file
!ls -lh "{workdir}"/*.yoda.gz
!yoda2root "{YODA_OUT}" 2>/dev/null && echo "Wrote ROOT file next to the YODA output" || echo "yoda2root not available - use the YODA file directly"

---
That covers the full workflow: find a sample, read its events, run an analysis and look at the results.
You can now swap in any other dataset, try a different Rivet analysis, or write your own routine.

### Learn more

- ATLAS Open Data: [event generation](https://opendata.atlas.cern/docs/data/for_research/evgen_data), [portal record](https://opendata.cern.ch/record/160000)
- [`atlasopenmagic`](https://opendata.atlas.cern/docs/atlasopenmagic): metadata and file access
- [Rivet](https://rivet.hepforge.org), [YODA](https://yoda.hepforge.org), [HepMC](https://hepmc.web.cern.ch)
- [What is SWAN?](https://swan.docs.cern.ch/intro/what_is/): one way to get a CVMFS environment

*Example dataset:* DSID **601189** `PhPy8EG_AZNLO_Zee`, POWHEG+Pythia8 Drell-Yan $Z\to ee$,
$\sqrt{s}=13.6$ TeV, $\sigma \approx 1998.8$ pb.

<div class="alert alert-block alert-info">
We welcome your feedback on this notebook or any of our other materials! Please <a href="https://forms.gle/zKBqS1opAHHemv9U7">fill out this survey</a> to let us know how we're doing, and you can enter a raffle to win some <a href="https://atlas-secretariat.web.cern.ch/merchandise">ATLAS merchandise</a>!
</div>